In [30]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================================================
# LOAD CSV DATA
# =========================================================

baca = pd.read_csv("baca_results.csv")
aca = pd.read_csv("aca_results.csv")

# =========================================================
# CREATE INTERACTIVE FIGURE
# =========================================================

fig = go.Figure()

# =========================================================
# BACA CURVES
# =========================================================

d_values = sorted(baca["d"].unique())

for d in d_values:

    df = baca[baca["d"] == d]
    df = df.sort_values("n")

    x = df["n"].to_numpy()

    elapsed = df["elapsed"].to_numpy()
    rank = df["rank"].to_numpy()

    y = elapsed / rank

    # -----------------------------------------------------
    # MEASURED DATA
    # -----------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode='markers',
            name=f'BACA d={d} measured'
        )
    )

    # -----------------------------------------------------
    # LINEAR FIT
    # -----------------------------------------------------

    coeff_lin = np.polyfit(x, y, 1)
    fit_lin = np.poly1d(coeff_lin)

    x_fit = np.linspace(
        np.min(x),
        np.max(x),
        300
    )

    fig.add_trace(
        go.Scatter(
            x=x_fit,
            y=fit_lin(x_fit),
            mode='lines',
            name=f'BACA d={d} linear fit'
        )
    )

    # -----------------------------------------------------
    # QUADRATIC FIT
    # -----------------------------------------------------

    coeff_quad = np.polyfit(x, y, 2)
    fit_quad = np.poly1d(coeff_quad)

    fig.add_trace(
        go.Scatter(
            x=x_fit,
            y=fit_quad(x_fit),
            mode='lines',
            name=f'BACA d={d} quadratic fit',
            visible='legendonly'  # hidden initially
        )
    )

    # -----------------------------------------------------
    # PRINT COEFFICIENTS
    # -----------------------------------------------------

    print("\n===================================")
    print(f"BACA d = {d}")
    print("===================================")

    print("Linear coefficients:")
    print(coeff_lin)

    print("\nQuadratic coefficients:")
    print(coeff_quad)

# =========================================================
# ACA DATA
# =========================================================

aca = aca.sort_values("n")

x_aca = aca["n"].to_numpy()

elapsed_aca = aca["elapsed"].to_numpy()
rank_aca = aca["rank"].to_numpy()

y_aca = elapsed_aca / rank_aca

# ---------------------------------------------------------
# ACA MEASURED
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x_aca,
        y=y_aca,
        mode='markers',
        name='ACA measured'
    )
)

# ---------------------------------------------------------
# ACA LINEAR FIT
# ---------------------------------------------------------

coeff_lin_aca = np.polyfit(x_aca, y_aca, 1)
fit_lin_aca = np.poly1d(coeff_lin_aca)

x_fit_aca = np.linspace(
    np.min(x_aca),
    np.max(x_aca),
    300
)

fig.add_trace(
    go.Scatter(
        x=x_fit_aca,
        y=fit_lin_aca(x_fit_aca),
        mode='lines',
        name='ACA linear fit'
    )
)

# ---------------------------------------------------------
# ACA QUADRATIC FIT
# ---------------------------------------------------------

coeff_quad_aca = np.polyfit(x_aca, y_aca, 2)
fit_quad_aca = np.poly1d(coeff_quad_aca)

fig.add_trace(
    go.Scatter(
        x=x_fit_aca,
        y=fit_quad_aca(x_fit_aca),
        mode='lines',
        name='ACA quadratic fit',
        visible='legendonly'
    )
)

# =========================================================
# PRINT ACA COEFFICIENTS
# =========================================================

print("\n===================================")
print("ACA")
print("===================================")

print("Linear coefficients:")
print(coeff_lin_aca)

print("\nQuadratic coefficients:")
print(coeff_quad_aca)

# =========================================================
# LAYOUT
# =========================================================

fig.update_layout(
    title="ACA vs BACA Runtime Scaling",
    xaxis_title="n",
    yaxis_title="elapsed / rank",
    template="plotly_white",
    hovermode="x unified",
    width=1200,
    height=700
)

# =========================================================
# SHOW
# =========================================================

fig.show()


BACA d = 2
Linear coefficients:
[-1.90840384e-06  8.40611127e-03]

Quadratic coefficients:
[-1.28297311e-08  3.78637627e-05 -2.13588650e-02]

BACA d = 4
Linear coefficients:
[ 6.79088334e-06 -4.63906084e-03]

Quadratic coefficients:
[-1.65244231e-08  5.80165948e-05 -4.29757223e-02]

BACA d = 8
Linear coefficients:
[ 1.75128823e-05 -1.35388345e-02]

Quadratic coefficients:
[-8.29800387e-09  4.32366943e-05 -3.27902035e-02]

ACA
Linear coefficients:
[ 1.38052320e-06 -8.94965132e-04]

Quadratic coefficients:
[ 8.66145110e-10 -1.30452664e-06  1.11449152e-03]


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# =========================================================
# LOAD DATA
# =========================================================

# expected CSV format:
#
# d,rank,frob
#
# example:
# 2,1,1.23e-2
# 2,2,8.31e-3
# ...

df = pd.read_csv(
    "frobenius_norms.csv",
    names=["d", "rank", "frob"]
)

# =========================================================
# CLEAN DATA
# =========================================================

df = df.sort_values(["d", "rank"])

d_values = sorted(df["d"].unique())

# =========================================================
# CREATE FIGURE
# =========================================================

fig = go.Figure()

# =========================================================
# PLOT EACH d
# =========================================================

for d in d_values:

    sub = df[df["d"] == d]

    x = sub["rank"].to_numpy()
    y = sub["frob"].to_numpy()

    # -----------------------------------------------------
    # RAW DATA
    # -----------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines+markers",
            name=f"d = {d}"
        )
    )

    # -----------------------------------------------------
    # LOG FIT
    # -----------------------------------------------------

    # remove zeros for log fit
    mask = y > 0

    x_fit_data = x[mask]
    y_fit_data = y[mask]

    if len(y_fit_data) > 2:

        logy = np.log(y_fit_data)

        coeff = np.polyfit(x_fit_data, logy, 1)

        fitted = np.exp(
            coeff[0] * x_fit_data + coeff[1]
        )

        fig.add_trace(
            go.Scatter(
                x=x_fit_data,
                y=fitted,
                mode="lines",
                line=dict(dash="dash"),
                name=f"d = {d} exp fit",
                visible="legendonly"
            )
        )

        print("\n===================================")
        print(f"d = {d}")
        print("===================================")

        print("Exponential decay fit:")
        print(f"log(error) = {coeff[0]:.6e} * rank + {coeff[1]:.6e}")

# =========================================================
# LAYOUT
# =========================================================

fig.update_layout(
    title="Frobenius Error vs Rank",
    xaxis_title="Rank",
    yaxis_title="||A - R||_F²",
    template="plotly_white",
    hovermode="x unified",
    width=1200,
    height=700
)

# =========================================================
# LOG SCALE
# =========================================================

fig.update_yaxes(type="log")

# =========================================================
# SHOW
# =========================================================

fig.show()

TypeError: '>' not supported between instances of 'str' and 'int'